In [0]:
%run .././utils/spark_config

In [0]:
import os
from datetime import datetime

# ── Build date-partitioned staging path ───────────────────────────
now        = datetime.now()
year       = now.strftime("%Y")
month      = now.strftime("%m")
day        = now.strftime("%d")

# Dataset folder + Hive-style date partition
dataset_folder   = "creditcardfraud"
partition_path   = f"year={year}/month={month}/day={day}/"
staged_file_path = (
    paths["staging"]
    + f"{dataset_folder}/{partition_path}creditcard.csv"
)

local_download_path = "/tmp/kaggle/"
local_file_path     = "file:///tmp/kaggle/creditcard.csv"

print("=" * 60)
print(" DATA SIMULATOR — CELL 2: STAGING INGESTION")
print("=" * 60)
print(f"📅 Partition   : year={year}/month={month}/day={day}")
print(f"📂 Target path : {staged_file_path}")

# ── Check if today's partition already exists in ADLS ─────────────
partition_exists = False

try:
    partition_folder = (
        paths["staging"]
        + f"{dataset_folder}/{partition_path}"
    )
    files = dbutils.fs.ls(partition_folder)
    data_files = [f for f in files if not f.name.startswith("_")
                                   and not f.name.startswith(".")]
    if len(data_files) > 0:
        partition_exists = True
        print(f"\n✅ Today's partition already exists — skipping download")
        print(f"   Files: {[f.name for f in data_files]}")
except Exception:
    partition_exists = False
    print(f"\n📥 Today's partition not found — starting download...")

# ── Download + copy only if partition does not exist ──────────────
if not partition_exists:

    # Fetch Kaggle credentials from Key Vault
    os.environ["KAGGLE_USERNAME"] = dbutils.secrets.get(
        "portfolio-scope", "kaggle-username")
    os.environ["KAGGLE_KEY"]      = dbutils.secrets.get(
        "portfolio-scope", "kaggle-key")

    # Step 1: Download raw file from Kaggle to cluster /tmp
    print("📥 Step 1: Downloading from Kaggle to cluster /tmp...")
    import kaggle
    kaggle.api.dataset_download_files(
        "mlg-ulb/creditcardfraud",
        path=local_download_path,
        unzip=True,
        quiet=False
    )
    print(f"✅ Download complete → {local_download_path}creditcard.csv")

    # Step 2: Copy raw file directly to date-partitioned ADLS path
    print(f"📋 Step 2: Copying raw file to date-partitioned staging path...")
    dbutils.fs.cp(
        local_file_path,    # Source: cluster local file
        staged_file_path,   # Destination: partitioned ADLS path
        recurse=False
    )
    print(f"✅ Raw file copied to ADLS")
    print(f"   Source      : {local_file_path}")
    print(f"   Destination : {staged_file_path}")
    print(f"   Method      : dbutils.fs.cp — zero Spark processing")

# ── Verify — confirm file in correct partition ────────────────────
print(f"\n🔍 Verifying staging partition in ADLS...")

partition_folder = (
    paths["staging"]
    + f"{dataset_folder}/{partition_path}"
)
files      = dbutils.fs.ls(partition_folder)
data_files = [f for f in files if not f.name.startswith("_")
                               and not f.name.startswith(".")]

for f in data_files:
    size_mb = round(f.size / (1024 * 1024), 2)
    print(f"✅ {f.name} — {size_mb} MB")

# ── Show full staging folder structure ────────────────────────────
print(f"\n📂 Full staging folder structure:")
print(f"   staging/")
print(f"   └── {dataset_folder}/")
print(f"       └── year={year}/")
print(f"           └── month={month}/")
print(f"               └── day={day}/")
print(f"                   └── creditcard.csv ✅")

print(f"\n✅ Staging ingestion complete")
print(f"   File preserved exactly as received from Kaggle")
print(f"   Single raw file — not split into Spark part files")
print(f"   Date partition  — year={year}/month={month}/day={day}")

# ── Store staged path for downstream cells ────────────────────────
# Cell 3 and Cell 4 will use this path
staged_today_path = staged_file_path
print(f"\n📌 staged_today_path set → {staged_today_path}")
print("\nCell 2 complete — ready for Cell 3")
